# Lab 02 Cheatsheet
**Medical Image Enhancement, Multimodal Fusion & Real-Time Video** | CV Practical Lab Exam

### Contents
| # | Section | # | Section |
|---|---|---|---|
| 0 | Setup & helper functions | 4 | Task 3 - Real-time echo video |
| 1 | Key concepts: `IMREAD_GRAYSCALE` vs `COLOR_BGR2RGB` | 5 | Extras (CLAHE, histograms, saving) |
| 2 | Task 1 - Chest X-ray enhancement | 6 | Function quick reference |
| 3 | Task 2 - CT + MRI fusion | 7 | Common errors & exam checklist |

**Pipeline order to remember (all three tasks):**
`load -> equalize -> colour-map -> channel balance / fuse -> log -> gamma -> display`

---
## 0. Setup & helper functions
Run this cell first. Change the three paths to match the exam files.

| Helper | Purpose |
|---|---|
| `load_gray(path)` | `cv2.imread(..., IMREAD_GRAYSCALE)` that **raises** if the file is missing (imread silently returns `None`) |
| `log_transform(img)` | `s = c * log(1 + r)`, rescaled to 0-255 uint8 |
| `gamma_transform(img, gamma)` | `s = 255 * (r/255) ** gamma`, uint8 |
| `show(images, titles)` | Side-by-side display. Gray arrays use a **fixed 0-255 scale**; colour arrays must already be **RGB** |

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# PATHS  (edit these)
# ============================================================
XRAY_PATH  = 'data/COVID/COVID_2.png'
CT_PATH    = 'data/synthetic_slices/Patient_001/CT/slice_001.png'
MRI_PATH   = 'data/synthetic_slices/Patient_001/MRI/slice_001.png'
VIDEO_PATH = 'data/EchoNet-Dynamic/Videos/sample.mp4'
# Relative paths are resolved from the NOTEBOOK's folder, not the project root.

# ============================================================
# HELPERS
# ============================================================
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)      # 1-channel uint8 (H, W)
    if img is None:                                   # imread does NOT raise on a bad path
        raise FileNotFoundError(path)
    return img

def log_transform(img):
    # expands dark values, compresses bright ones -> reveals faint structure
    out = np.log1p(img.astype(np.float32))
    return cv2.normalize(out, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def gamma_transform(img, gamma=0.5):
    # gamma < 1 brightens dark areas, gamma > 1 darkens
    out = np.power(img.astype(np.float32) / 255.0, gamma)
    return (out * 255).astype(np.uint8)

def show(images, titles=None, figsize=(15, 5)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    axes = np.atleast_1d(axes)
    for i, (ax, im) in enumerate(zip(axes, images)):
        if im.ndim == 2:
            ax.imshow(im, cmap='gray', vmin=0, vmax=255)   # fixed range: no auto-stretch
        else:
            ax.imshow(im)                                  # must be RGB
        if titles:
            ax.set_title(titles[i])
        ax.axis('off')
    plt.tight_layout()
    plt.show()

---
# 1. Key concepts: `IMREAD_GRAYSCALE` vs `COLOR_BGR2RGB`

`cv2.IMREAD_GRAYSCALE` is a flag for `cv2.imread`. It loads the file as a single-channel 2D array (H×W) with values 0–255.

- Use it when color carries no information: CT, MRI and X-ray images, or any pipeline that needs one intensity channel.
- It is required for `cv2.equalizeHist`, which only accepts single-channel 8-bit images, and for `applyColorMap`, which expects that kind of input.
- If you load a file without it, OpenCV gives you a 3-channel BGR array, even for a gray image.

`cv2.COLOR_BGR2RGB` is a conversion code for `cv2.cvtColor`. It swaps the channel order of a 3-channel image.

- OpenCV stores color in BGR order. Matplotlib, PIL and most other libraries expect RGB.
- Use it when you display, save or pass a color image from OpenCV to one of those libraries. Examples are `plt.imshow` on a color image loaded with `imread` (default flag) or produced by `applyColorMap`.
- Don't use it on single-channel images. They have no channel order to swap, and the call either fails or turns them into 3 channels. That was the cell 1 bug earlier.
- Don't use it between OpenCV functions. `addWeighted`, `resize` and the like work the same in any order, so convert once, right before you display.

**Rule of thumb:** ask how many channels the array has. If it's 1, display it with `cmap='gray'`. If it's 3 and it came from OpenCV, convert BGR to RGB right before showing it.

### Why `vmin=0, vmax=255` matters
`plt.imshow` on a gray image maps the image's *own* min to black and max to white. A dim image (e.g. range 0-130) is stretched and looks brighter than it is, and an image that is already full-range (e.g. after `equalizeHist`) looks unchanged. Pin the range so before/after comparisons are honest.

### Why `imread` "errors" show up somewhere else
`cv2.imread` and `cv2.VideoCapture` do not raise on a wrong path. They return `None` / `isOpened() == False`. The error then appears later (`imshow` of `None` -> *dtype object cannot be converted to float*; `equalizeHist` -> *Assertion failed*). **Always validate right after loading.**

In [ ]:
# ---------- Channel sanity-check template ----------
img_color = np.zeros((4, 4, 3), np.uint8)
img_gray  = np.zeros((4, 4), np.uint8)
print(img_color.shape, '-> 3 channels, from OpenCV: convert BGR->RGB before plt.imshow')
print(img_gray.shape,  '-> 1 channel: plt.imshow(..., cmap="gray"), NO cvtColor')

# Correct colour conversion (3-channel only)
rgb = cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB)
# gray -> 3-channel (only if a later step needs 3 channels)
gray3 = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2BGR)
# 3-channel -> gray
gray1 = cv2.cvtColor(img_color, cv2.COLOR_BGR2GRAY)

---
# 2. Task 1 - Chest X-ray enhancement
**Goal:** improve the visibility of lung structures and dense tissue.

| Step | Function | Purpose |
|---|---|---|
| Load | `cv2.imread(path, cv2.IMREAD_GRAYSCALE)` | single-channel intensity image |
| Equalize | `cv2.equalizeHist(img)` | spreads gray levels over 0-255 to boost contrast |
| Heatmap | `cv2.applyColorMap(eq, cv2.COLORMAP_JET)` | gray -> **BGR** colour map (3-channel) |
| Channel balance | `cv2.split` / `cv2.convertScaleAbs(ch, alpha, beta)` / `cv2.merge` | scale each channel (`alpha` = gain/contrast, `beta` = offset/brightness) |
| Dense tissue mask | `cv2.threshold(eq, 180, 255, THRESH_BINARY)` + `cv2.bitwise_and(..., mask=mask)` | keep only pixels brighter than the threshold |
| Log | `log1p` + `normalize` | brighten faint structure in dark regions |
| Gamma | `(r/255)**gamma * 255` | non-linear brightness; gamma < 1 brightens |

`convertScaleAbs(src, alpha, beta)` computes `saturate_uint8(|src*alpha + beta|)`.
`cv2.threshold` returns **two** values: `(threshold_used, binary_image)`.

In [ ]:
# ---------- TEMPLATE: X-ray enhancement pipeline ----------
img = load_gray(XRAY_PATH)                                  # 1. load (grayscale)

equalized = cv2.equalizeHist(img)                           # 2. histogram equalization

heatmap = cv2.applyColorMap(equalized, cv2.COLORMAP_JET)    # 3. colour map (BGR output)

b, g, r = cv2.split(heatmap)                                # 4. channel balancing
b = cv2.convertScaleAbs(b, alpha=1.0,  beta=0)
g = cv2.convertScaleAbs(g, alpha=1.05, beta=0)
r = cv2.convertScaleAbs(r, alpha=0.95, beta=0)
balanced = cv2.merge([b, g, r])                             # still BGR

_, mask = cv2.threshold(equalized, 180, 255, cv2.THRESH_BINARY)   # 5. dense-tissue mask
dense_tissue = cv2.bitwise_and(equalized, equalized, mask=mask)

log_img   = log_transform(img)                              # 6. log transform
gamma_img = gamma_transform(img, gamma=0.5)                 # 7. gamma correction

show([img, equalized, dense_tissue],
     ['Original', 'Equalized', 'Dense tissue mask'])
show([cv2.cvtColor(heatmap,  cv2.COLOR_BGR2RGB),            # convert ONCE, right before display
      cv2.cvtColor(balanced, cv2.COLOR_BGR2RGB)],
     ['Heatmap (JET)', 'Balanced heatmap'], figsize=(10, 5))
show([img, log_img, gamma_img], ['Original', 'Log', 'Gamma 0.5'])

---
# 3. Task 2 - CT + MRI multimodal fusion
**Goal:** combine a CT slice (sharp anatomical edges) with its MRI slice (internal tissue contrast) into one view.

**Order matters:** equalize **each** modality independently -> colour map -> weighted fusion -> log / gamma on the **fused** result -> side-by-side comparison.

| Function | Notes |
|---|---|
| `cv2.addWeighted(src1, alpha, src2, beta, gamma)` | `dst = src1*alpha + src2*beta + gamma`. Both images must have the **same shape and dtype**. Use `alpha + beta = 1`. |
| `cv2.resize(img, (width, height))` | dsize is **(width, height)** - the reverse of `.shape` |

**Weights:** the task asks for a *heavier CT weight* (keeps sharp edges) and a *lighter MRI weight* (highlights tissue). e.g. `CT 0.7`, `MRI 0.3`.
Both inputs are 3-channel BGR at fusion time because `applyColorMap` returns BGR.

**Standalone frames** in the final comparison = the original (or equalized) CT and MRI, **not** the log/gamma versions.

In [ ]:
# ---------- TEMPLATE: CT + MRI fusion ----------
ct  = load_gray(CT_PATH)                                    # 1. load both modalities (grayscale)
mri = load_gray(MRI_PATH)

ct_eq  = cv2.equalizeHist(ct)                               # 2. equalize each independently
mri_eq = cv2.equalizeHist(mri)

ct_heat  = cv2.applyColorMap(ct_eq,  cv2.COLORMAP_BONE)     # 3. colour map (BGR)
mri_heat = cv2.applyColorMap(mri_eq, cv2.COLORMAP_JET)      #    e.g. CT in bone, MRI in JET

if ct_heat.shape != mri_heat.shape:                         # 4. shapes must match for addWeighted
    mri_heat = cv2.resize(mri_heat, (ct_heat.shape[1], ct_heat.shape[0]))  # (width, height)

fused = cv2.addWeighted(ct_heat, 0.7, mri_heat, 0.3, 0)     # 5. weighted fusion: heavier CT

fused_log   = log_transform(fused)                          # 6. log then gamma on the FUSED image
fused_final = gamma_transform(fused_log, gamma=0.5)

# 7. comparison: standalone CT, standalone MRI, fused output
show([ct, mri, cv2.cvtColor(fused_final, cv2.COLOR_BGR2RGB)],
     ['CT', 'MRI', 'Fused (log + gamma)'])

### Alternative: matplotlib overlay (no `addWeighted`)
Useful to show MRI as a transparent heatmap over the CT.

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(ct, cmap='gray', vmin=0, vmax=255)
plt.imshow(mri, cmap='jet', alpha=0.4, vmin=0, vmax=255)   # alpha = transparency of the top layer
plt.axis('off')
plt.title('CT + MRI overlay')
plt.show()

---
# 4. Task 3 - Real-time echo video

**Goal:** apply the same enhancement pipeline to every frame of an echocardiogram video and show raw vs enhanced side by side.

| Function | What it does |
|---|---|
| `cv2.VideoCapture(path)` | Opens a video. **Does not raise** on a bad path; check `cap.isOpened()`. |
| `cap.read()` | Returns `(ret, frame)`. `ret == False` at the end of the video. `frame` is **BGR, 3-channel** |
| `cap.get(cv2.CAP_PROP_FPS)` / `CAP_PROP_FRAME_COUNT` | Video metadata |
| `cap.release()` | Frees the file. **After release, `cap.read()` returns `False` immediately** - re-run the cell that opens the video before replaying |
| `cv2.hconcat([a, b])` | Joins images horizontally. Same height **and** same channel count required |
| `cv2.imshow` / `cv2.waitKey` | Opens an OpenCV window. Works in a plain `.py` script, **not reliably in Jupyter** |

**Frames are colour** (3-channel), so convert to gray first: `cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)` -> then `equalizeHist` -> `applyColorMap`.

**In Jupyter**, `plt.imshow` inside a loop only draws when the cell ends. Use `clear_output(wait=True)` + `plt.show()` each iteration to get an animation.

In [ ]:
# ---------- TEMPLATE (Jupyter): real-time processing ----------
from IPython.display import clear_output
import time

cap = cv2.VideoCapture(VIDEO_PATH)                          # re-run THIS line before every replay
if not cap.isOpened():
    raise RuntimeError(f'Could not open video: {VIDEO_PATH}')
fps = cap.get(cv2.CAP_PROP_FPS) or 25

def enhance_frame(frame, gamma=0.8):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)          # frame is BGR -> 1 channel
    eq   = cv2.equalizeHist(gray)
    heat = cv2.applyColorMap(eq, cv2.COLORMAP_JET)          # BGR
    b, g, r = cv2.split(heat)
    b = cv2.convertScaleAbs(b, alpha=1.0)
    g = cv2.convertScaleAbs(g, alpha=1.05)
    r = cv2.convertScaleAbs(r, alpha=0.95)
    balanced = cv2.merge([b, g, r])
    return gamma_transform(log_transform(balanced), gamma)

while True:
    ret, frame = cap.read()
    if not ret:                                             # end of video
        break

    enhanced = enhance_frame(frame)
    combined = cv2.hconcat([frame, enhanced])               # raw | enhanced

    clear_output(wait=True)
    plt.figure(figsize=(10, 5))
    plt.imshow(cv2.cvtColor(combined, cv2.COLOR_BGR2RGB))   # convert once, at display
    plt.axis('off')
    plt.show()
    time.sleep(1 / fps)                                     # play at real speed

cap.release()

In [ ]:
# ---------- TEMPLATE (plain .py script): OpenCV window ----------
# Run from a terminal, NOT inside Jupyter. Press 'q' to quit.
#
# cap = cv2.VideoCapture(VIDEO_PATH)
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break
#     combined = cv2.hconcat([frame, enhance_frame(frame)])
#     cv2.imshow('Raw vs Enhanced', combined)               # no BGR->RGB needed: OpenCV expects BGR
#     if cv2.waitKey(1) & 0xFF == ord('q'):
#         break
# cap.release()
# cv2.destroyAllWindows()

---
# 5. Extras likely to help in the exam

### 5.1 Histogram before / after equalization
Equalization only has something to spread if the image has several gray levels. A 2-level synthetic image just gets its two values remapped.

In [ ]:
def show_hist(img_before, img_after, titles=('Before', 'After')):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, im, t in zip(axes, (img_before, img_after), titles):
        ax.hist(im.ravel(), bins=256, range=(0, 255))
        ax.set_title(t)
    plt.tight_layout()
    plt.show()

# show_hist(img, equalized, ('Original', 'Equalized'))
print('unique gray levels:', len(np.unique(img)) if 'img' in dir() else 'load an image first')

### 5.2 CLAHE (adaptive equalization)
Equalizes small tiles with a clip limit, so noise is not over-amplified. Often better than global `equalizeHist` for X-ray / ultrasound.

In [ ]:
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
# img_clahe = clahe.apply(img)        # img must be single-channel uint8

### 5.3 Saving results
`cv2.imwrite` expects **BGR** (the OpenCV order). Do **not** convert to RGB before saving.

In [ ]:
# cv2.imwrite('fused_output.png', fused_final)       # BGR array -> saved with correct colours

# ----- save the processed video -----
# cap = cv2.VideoCapture(VIDEO_PATH)
# w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
# out = cv2.VideoWriter('enhanced.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
# while True:
#     ret, frame = cap.read()
#     if not ret: break
#     out.write(enhance_frame(frame))                 # frame size must equal (w, h), 3-channel
# out.release(); cap.release()

---
# 6. Function quick reference

| Function | Input -> Output | Notes |
|---|---|---|
| `cv2.imread(p, cv2.IMREAD_GRAYSCALE)` | file -> `(H,W)` uint8 | `None` if path is wrong |
| `cv2.equalizeHist(g)` | `(H,W)` uint8 -> `(H,W)` uint8 | **1-channel 8-bit only** |
| `cv2.applyColorMap(g, cmap)` | `(H,W)` uint8 -> `(H,W,3)` **BGR** | `COLORMAP_JET`, `_HOT`, `_BONE`, `_TURBO`, `_VIRIDIS` |
| `cv2.split(bgr)` / `cv2.merge([b,g,r])` | split / combine channels | order is **B, G, R** |
| `cv2.convertScaleAbs(x, alpha, beta)` | `abs(x*alpha + beta)` clipped to uint8 | alpha = gain, beta = offset |
| `cv2.threshold(g, t, 255, THRESH_BINARY)` | returns **`(t, mask)`** | pixels > t become 255 |
| `cv2.bitwise_and(a, a, mask=m)` | keeps pixels where mask != 0 | |
| `cv2.addWeighted(a, α, b, β, γ)` | `a*α + b*β + γ` | same shape and dtype |
| `cv2.normalize(x, None, 0, 255, NORM_MINMAX)` | rescales to range | cast with `.astype(np.uint8)` afterwards |
| `cv2.cvtColor(x, code)` | `BGR2RGB`, `BGR2GRAY`, `GRAY2BGR` | match code to the array's channels |
| `cv2.resize(x, (w, h))` | | dsize is (width, height) |
| `cv2.hconcat` / `cv2.vconcat` | join images | same height / width **and** channels |
| `np.log1p(x.astype(np.float32))` | `log(1 + x)` | cast first, or uint8 overflows |
| `np.power(x/255.0, gamma)*255` | gamma correction | normalise to 0-1 first |
| `cv2.VideoCapture(p)` / `.read()` / `.release()` | video I/O | check `.isOpened()` |

**Transformations**

| Transform | Formula | Effect |
|---|---|---|
| Log | `s = c * log(1 + r)` | Expands dark values, compresses bright ones |
| Power-law / gamma | `s = 255 * (r/255)^gamma` | `gamma < 1` brightens, `gamma > 1` darkens |
| Histogram equalization | CDF remapping | Spreads gray levels to the full 0-255 range |

---
# 7. Common errors & exam checklist

| Symptom / error | Cause | Fix |
|---|---|---|
| `TypeError: Image data of dtype object cannot be converted to float` | `imread` returned `None` (wrong path) | Fix path; add `if img is None: raise` |
| `Assertion failed ... _src.type() == CV_8UC1` in `equalizeHist` | Image has 3 channels (loaded without `IMREAD_GRAYSCALE`, or wrongly run through `cvtColor`) | Load as grayscale / `cvtColor(..., COLOR_BGR2GRAY)` |
| Colours look wrong (red <-> blue) | BGR displayed as RGB | `cvtColor(..., COLOR_BGR2RGB)` once, before `plt.imshow` |
| `cmap='gray'` has no effect | Image is 3-channel; `cmap` is ignored for RGB data | Only use `cmap` on 2-D arrays |
| `addWeighted` shape / type error | Images differ in size or channels | `resize`, make both 3-channel |
| Original and equalized images look the same | `imshow` auto-stretches; or image only has 2 gray levels | `vmin=0, vmax=255`; check `len(np.unique(img))` |
| `Could not open video` | Wrong relative path | Path is relative to the **notebook's** folder |
| Video loop shows nothing on re-run | `cap` was already released | Re-run the `VideoCapture(...)` cell |
| `cv2.imshow` window never appears / kernel hangs in Jupyter | OpenCV GUI vs notebook | Use `plt` + `clear_output(wait=True)` |
| `hconcat` error | Different heights or channel counts | Make both 3-channel and same size |
| Log / gamma output looks black or saturated | Forgot to normalise / cast to `uint8` | `normalize(...,0,255).astype(np.uint8)`; divide by 255 before `power` |

### Exam checklist
1. Load -> **validate** (`None` check / `isOpened`).
2. Grayscale for CT / MRI / X-ray / echo frames.
3. `equalizeHist` on **single-channel** images, each modality independently.
4. `applyColorMap` -> result is **BGR**.
5. Fuse with `addWeighted` (same shape; CT heavier, MRI lighter if asked).
6. Log and gamma on the **fused** result, always cast back to `uint8`.
7. Convert **BGR -> RGB once**, only at display time, only for 3-channel arrays.
8. Show original / standalone frames next to the result, with titles, `axis('off')`, `vmin=0, vmax=255`.